# 13. Final evaluation, part A: ensemble freeze

This is the first of the two-part final evaluation. Here the deployed **M3 + M4 rank-vote** ensemble is assembled and **frozen on folds 1 to 9**. The second half, `13_final_evaluation_fold10.ipynb`, performs the single atomic **fold10** test-set contact. Ensemble-freeze and fold10-contact are deliberately kept as two separate halves. See `JOURNAL.md` (the ensemble rank-vote and single fold10-contact decisions).

# Main ensemble (deployed model: M3 + M4 rank-vote)

**Read-only, single Run All.** Loads OOF scores from disk only; no model prediction, no refit. **fold10 is NOT touched anywhere in this notebook.** The deployed model is a **rank-vote on OOF percentiles** (rank transfers across datasets; raw score scale does not - this is what makes the ensemble robust to the batch effect).

`alpha` is chosen on **OOF folds 1-8 only** (never fold9/10). Threshold = F1-max on the OOF ensemble score. The frozen artifact is self-contained so the final evaluation can rank fold10 scores against the frozen folds-1-8 reference and apply alpha+threshold with **zero new decisions**. `ecg_id` as str, align on `(ecg_id, source)`, M4 = `m4_combined_oof.csv` (never wavelet_env).

## Section 1 - load + align + OOF percentile ranks

In [ ]:
# ===================== SECTION 1 — load + align + OOF percentile ranks =====================
# Read-only: OOF scores from disk. No model prediction, no refit. fold10 NEVER touched anywhere in this notebook.
# Deployed ensemble = rank-vote on OOF percentiles (rank transfers cross-dataset; raw score scale does not = batch effect).
import os, sys, json, warnings, datetime
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROC=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src'); FIGDIR=os.path.join(ROOT,'reports','figures')
ENSDIR=os.path.join(ROOT,'models','ensemble'); os.makedirs(FIGDIR,exist_ok=True); os.makedirs(ENSDIR,exist_ok=True)
sys.path.insert(0,SRC); from evaluation import _boot_ci, f1max_threshold
SECTIONS={}
CANON=['ecg_id','source','fold','label','proba_raw','proba_cal']; KEY=['ecg_id','source']
def load(fn):
    d=pd.read_csv(os.path.join(PROC,fn),dtype={'ecg_id':str}); assert list(d.columns)==CANON,'non-canonical '+fn; return d
try:
    M3f='m3_combined_oof.csv'; M4f='m4_combined_oof.csv'          # M4 = canonical (NOT wavelet_env)
    M3=load(M3f); M4=load(M4f)
    print('M3 folds present:',sorted(M3.fold.unique()),'| M4 folds present:',sorted(M4.fold.unique()))
    assert set(M3.fold.unique())<=set(range(1,9)) and set(M4.fold.unique())<=set(range(1,9)), 'fold 9/10 rows present -> STOP'
    D=(M3[['ecg_id','source','fold','label','proba_raw']].rename(columns={'proba_raw':'m3'})
       .merge(M4[['ecg_id','source','proba_raw']].rename(columns={'proba_raw':'m4'}),on=KEY,how='inner'))
    assert len(D)==53540 and int((D.label==1).sum())==115, 'align FAIL: %d rows %d WPW'%(len(D),int((D.label==1).sum()))
    ap_m3=float(average_precision_score(D.label,D.m3)); ap_m4=float(average_precision_score(D.label,D.m4))
    print('Aligned %d rows / %d WPW / folds 1-8 only.'%(len(D),int((D.label==1).sum())))
    print('OOF AP recomputed: M3=%.4f (ref 0.619) | M4=%.4f (ref 0.718)'%(ap_m3,ap_m4))
    assert abs(ap_m3-0.619)<0.012 and abs(ap_m4-0.718)<0.012, 'OOF AP deviates from frozen -> STOP'
    D['r3']=D.m3.rank(pct=True); D['r4']=D.m4.rank(pct=True)     # OOF percentile ranks (folds 1-8 only)
    yb=D.label.values.astype(int)
    print('Section 1 OK.'); SECTIONS['S1']='OK'
except Exception as e:
    SECTIONS['S1']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 2 - alpha weight sweep on OOF (folds 1-8)

In [ ]:
# ===================== SECTION 2 — alpha weight sweep on OOF (folds 1-8 ONLY) =====================
try:
    import matplotlib.pyplot as plt
    alphas=np.round(np.arange(0.0,1.0001,0.02),2)
    def ens_score(a): return a*D.r4.values+(1-a)*D.r3.values
    aps=np.array([average_precision_score(yb,ens_score(a)) for a in alphas])
    i_eq=int(np.where(np.isclose(alphas,0.5))[0][0]); i_am=int(np.argmax(aps))
    a_am=float(alphas[i_am]); ap_eq=float(aps[i_eq]); ap_am=float(aps[i_am]); ap_m3o=float(aps[0]); ap_m4o=float(aps[-1])
    lo_eq,hi_eq=_boot_ci(yb,ens_score(0.5),average_precision_score)
    lo_am,hi_am=_boot_ci(yb,ens_score(a_am),average_precision_score)
    tab=pd.DataFrame([dict(setting='alpha=0 (M3 alone)',alpha=0.0,OOF_AP=round(ap_m3o,4)),
                      dict(setting='alpha=0.5 (equal)',alpha=0.5,OOF_AP=round(ap_eq,4)),
                      dict(setting='alpha=1 (M4 alone)',alpha=1.0,OOF_AP=round(ap_m4o,4)),
                      dict(setting='argmax alpha',alpha=a_am,OOF_AP=round(ap_am,4))])
    print(tab.to_string(index=False))
    print('  equal-weight OOF AP %.4f CI95[%.4f,%.4f]'%(ap_eq,lo_eq,hi_eq))
    print('  argmax(alpha=%.2f) OOF AP %.4f CI95[%.4f,%.4f]'%(a_am,ap_am,lo_am,hi_am))
    fig,ax=plt.subplots(figsize=(8,5)); ax.plot(alphas,aps,'-',color='#2563eb')
    ax.axvline(0.5,ls='--',color='gray',label='equal weight'); ax.axvline(a_am,ls=':',color='#dc2626',label='argmax=%.2f'%a_am)
    ax.set_xlabel('alpha (weight on M4)'); ax.set_ylabel('ensemble OOF AP'); ax.set_title('M3+M4 rank-vote: OOF AP vs alpha (folds 1-8)')
    ax.legend(); ax.grid(alpha=.3)
    p=os.path.join(FIGDIR,'ensemble_alpha_sweep.png'); plt.tight_layout(); plt.savefig(p,dpi=160,bbox_inches='tight'); plt.show(); print('figure ->',p)
    print('Section 2 OK.'); SECTIONS['S2']='OK'
except Exception as e:
    SECTIONS['S2']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 3 - parsimony decision (pre-registered rule)

In [ ]:
# ===================== SECTION 3 — parsimony decision (pre-registered rule) =====================
try:
    # RULE (stated before looking): adopt argmax-alpha ONLY if its OOF AP exceeds equal-weight OOF AP by more than
    # HALF the bootstrap CI half-width; otherwise keep alpha=0.5 (parsimony).
    half_width=(hi_eq-lo_eq)/2.0; gain=ap_am-ap_eq; margin=0.5*half_width
    adopt=bool(gain>margin and not np.isclose(a_am,0.5))
    chosen_alpha=a_am if adopt else 0.5
    print('=== PARSIMONY DECISION ===')
    print('  equal-weight OOF AP = %.4f  | argmax(alpha=%.2f) OOF AP = %.4f'%(ap_eq,a_am,ap_am))
    print('  gain(argmax - equal) = %+.4f | bootstrap CI half-width = %.4f | margin(0.5*half) = %.4f'%(gain,half_width,margin))
    print('  adopt argmax ? %s  ->  FROZEN alpha = %.2f  (%s)'%(adopt,chosen_alpha,'argmax beats equal beyond margin' if adopt else 'within noise -> equal weight, parsimony'))
    print('Section 3 OK.'); SECTIONS['S3']='OK'
except Exception as e:
    SECTIONS['S3']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 4 - frozen ensemble artifact

In [ ]:
# ===================== SECTION 4 — frozen ensemble artifact =====================
try:
    ens=chosen_alpha*D.r4.values+(1-chosen_alpha)*D.r3.values
    ap=float(average_precision_score(yb,ens)); auc=float(roc_auc_score(yb,ens)); lo,hi=_boot_ci(yb,ens,average_precision_score)
    thr=float(f1max_threshold(yb,ens))                          # F1-max threshold on the OOF ensemble rank-vote score
    pred=(ens>=thr).astype(int); tp=int(((pred==1)&(yb==1)).sum()); fp=int(((pred==1)&(yb==0)).sum()); fn=int(((pred==0)&(yb==1)).sum())
    # save frozen reference score arrays (folds 1-8 raw M3/M4) for ranking NEW data (e.g. fold10) against the frozen distribution
    np.save(os.path.join(ENSDIR,'ref_scores_M3.npy'),np.sort(D.m3.values))
    np.save(os.path.join(ENSDIR,'ref_scores_M4.npy'),np.sort(D.m4.values))
    cfg={'model':'ensemble_M3_M4_rankvote','members':['M3','M4'],'alpha_weight_on_M4':round(chosen_alpha,2),
         'fusion':'ens = alpha*pct(M4) + (1-alpha)*pct(M3)  on OOF percentile ranks',
         'oof_files':{'M3':'data/processed/m3_combined_oof.csv','M4':'data/processed/m4_combined_oof.csv'},
         'oof_folds':'1-8','n_oof':int(len(D)),'n_wpw_oof':115,
         'oof_AP':round(ap,4),'oof_AP_CI95':[round(lo,4),round(hi,4)],'oof_AUC':round(auc,4),
         'threshold_F1max_on_OOF_ensemble':round(thr,4),
         'oof_confusion_at_threshold':{'TP':tp,'FP':fp,'FN':fn},
         'ref_scores':{'M3':'models/ensemble/ref_scores_M3.npy','M4':'models/ensemble/ref_scores_M4.npy'},
         'ranking_recipe':('To score NEW data (e.g. fold10) with ZERO new decisions: for each raw score x, '
            'pct_M = searchsorted(sorted ref_scores_M, x)/len(ref)  (fraction of frozen folds-1-8 scores <= x); '
            'ens = alpha*pct_M4 + (1-alpha)*pct_M3 ; flag WPW if ens >= threshold. '
            'Rank against the frozen reference ONLY - do NOT re-rank jointly with the new data.'),
         'date_frozen':datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),'fold10':'UNTOUCHED (the final evaluation)'}
    json.dump(cfg,open(os.path.join(ENSDIR,'ensemble_config.json'),'w',encoding='utf-8'),ensure_ascii=False,indent=2)
    print('FROZEN ensemble: alpha=%.2f | OOF AP %.4f CI95[%.4f,%.4f] | AUC %.4f | thr(F1max)=%.4f'%(chosen_alpha,ap,lo,hi,auc,thr))
    print('  OOF confusion @thr: TP %d FP %d FN %d'%(tp,fp,fn))
    print('  artifact -> models/ensemble/ensemble_config.json (+ ref_scores_M3/M4.npy)')
    print('Section 4 OK.'); SECTIONS['S4']='OK'
except Exception as e:
    SECTIONS['S4']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 4b - standardized evaluation (OOF)

In [ ]:
# ===================== SECTION 4b — standardized evaluation (OOF) =====================
# Run the frozen ensemble through evaluate_standard like every other model. OOF-only (test == OOF, folds 1-8).
# Does NOT recompute alpha or threshold (reuses the frozen ones from Section 4). fold10 is NOT touched.
try:
    from evaluation import evaluate_standard
    METRICS=os.path.join(ROOT,'reports','metrics'); os.makedirs(METRICS,exist_ok=True)
    evaluate_standard(
        name='ensemble_oof',
        y_oof=yb, score_oof=ens,            # folds 1-8 labels + ensemble rank-vote score
        y_test=yb, score_test=ens,          # OOF-only: test == OOF (no fold9/10 here)
        figures_dir=FIGDIR, metrics_dir=METRICS,
        ap_train=None, ap_oof=float(ap),    # true OOF AP -> gap_train_oof stays None cleanly
        multiseed=None,
        extra={'members':['M3','M4'],'alpha':float(chosen_alpha),'fusion':'rank-vote on OOF percentiles',
               'threshold_F1max':float(thr),
               'note':'OOF-only standardized card; fold10 evaluation is separate (the final evaluation, name=ensemble_fold10)'})
    print('\nThis is the OOF standardized card (folds 1-8). The held-out fold10 evaluation will be produced separately in '
          'the final evaluation as `ensemble_fold10` - this notebook never touches fold10.')
    print('-> reports/figures/ensemble_oof_evaluation.png , reports/metrics/ensemble_oof_metrics.json')
    print('Section 4b OK.'); SECTIONS['S4b']='OK'
except Exception as e:
    SECTIONS['S4b']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 5 - fold9 directional validation (looks once, decides nothing)

In [ ]:
# ===================== SECTION 5 — fold9 directional validation (looks once, decides NOTHING) =====================
try:
    f9_files=[f for f in ['m3_fold9_scores.csv','m4_fold9_scores.csv'] if os.path.exists(os.path.join(PROC,f))]
    has_f9=(9 in set(M3.fold.unique())) or (9 in set(M4.fold.unique())) or len(f9_files)==2
    if not has_f9:
        print('fold9 scores NOT available on disk (combined OOF = folds 1-8 only; fold9 needs frozen models = the final evaluation).')
        print('-> fold9 directional check SKIPPED. This validates nothing and changes nothing (alpha/threshold already frozen on OOF).')
        SECTIONS['S5']='SKIPPED (no fold9 scores on disk)'
    else:
        print('fold9 scores found -> directional sanity only (NOT used to choose alpha or threshold).')
        SECTIONS['S5']='OK (directional only)'
except Exception as e:
    SECTIONS['S5']='FAIL: %s'%e; import traceback; traceback.print_exc()

## Section 6 - summary

In [ ]:
# ===================== SECTION 6 — summary =====================
print('='*66); print('FROZEN DEPLOYED ENSEMBLE — M3 + M4 rank-vote'); print('='*66)
try:
    print('  members            : M3, M4 (OOF percentile rank-vote)')
    print('  alpha (weight M4)  : %.2f'%chosen_alpha)
    print('  F1-max threshold   : %.4f  (on OOF ensemble score)'%thr)
    print('  OOF AP (folds 1-8) : %.4f  CI95[%.4f,%.4f]'%(ap,lo,hi))
    print('  OOF AUC            : %.4f'%auc)
    print('  artifact           : models/ensemble/ensemble_config.json (+ ref_scores_M3/M4.npy)')
    print('  fold10             : UNTOUCHED. the final evaluation loads THIS artifact for the single test-set contact (zero new decisions).')
except Exception:
    print('  [summary unavailable - check earlier sections]')
print('\nSECTIONS:',{k:SECTIONS.get(k) for k in ['S1','S2','S3','S4','S4b','S5']})
print('Section 6 OK.')